In [ ]:
import spacy
import scispacy
from scispacy.linking import EntityLinker
import pandas as pd
from collections import Counter
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings("ignore")

print("🔹 正在加载 en_ner_bc5cdr_md 模型...")
nlp = spacy.load("en_ner_bc5cdr_md")

print("🔹 正在加载 UMLS Linker（标准化实体）...")
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
linker = nlp.get_pipe("scispacy_linker")

print("✅ 模型和 UMLS 加载成功")

INPUT_FILE = "./mimic_notes_final_clean.csv"
df = pd.read_csv(INPUT_FILE)
texts = df.iloc[:, 0].dropna().astype(str).tolist()
print(f"📄 共载入 {len(texts)} 条病历")

SAVE_EVERY = 1000
OUTPUT_FILE = "./disease_frequency_umls_incremental.csv"

disease_counter = Counter()
if os.path.exists(OUTPUT_FILE):
    print(f"🔁 检测到已有结果文件 {OUTPUT_FILE}，将继续累加计数。")
    prev_df = pd.read_csv(OUTPUT_FILE)
    for _, row in prev_df.iterrows():
        disease_counter[row["Disease"]] += int(row["Count"])

for i, text in enumerate(tqdm(texts, desc="Extracting and linking diseases"), start=1):
    doc = nlp(text[:3000])

    for ent in doc.ents:
        if ent.label_ == "DISEASE":
            if ent._.kb_ents:
                cui, score = ent._.kb_ents[0]
                disease_name = linker.kb.cui_to_entity[cui].canonical_name.lower()
            else:
                disease_name = ent.text.lower()

            disease_counter[disease_name] += 1

    if i % SAVE_EVERY == 0 or i == len(texts):
        temp_df = pd.DataFrame(disease_counter.most_common(), columns=["Disease", "Count"])
        temp_df.to_csv(OUTPUT_FILE, index=False)

        print(f"\n💾 已保存进度：{i}/{len(texts)} 条病历 → {OUTPUT_FILE}")
        print("📊 当前最常见疾病 Top 20:")
        for disease, count in disease_counter.most_common(20):
            print(f"  {disease:40s} {count}")
        print("-" * 60)

print("\n✅ 全部完成！最终疾病Top 20如下：")
for disease, count in disease_counter.most_common(20):
    print(f"{disease:40s} {count}")
print("\n📁 最终结果已保存：", OUTPUT_FILE)

In [ ]:
import spacy
import scispacy
from scispacy.linking import EntityLinker
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print("🔹 正在加载 en_ner_bc5cdr_md 模型和UMLS Linker...")
nlp = spacy.load("en_ner_bc5cdr_md")
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
linker = nlp.get_pipe("scispacy_linker")
print("✅ 模型和UMLS加载成功")

COPD_CUI = "C0024117"  # Chronic Obstructive Airway Disease
COPD_CUI_ALT = "C0008677"  # Chronic Obstructive Pulmonary Disease

TARGET_CUIS = {COPD_CUI, COPD_CUI_ALT}  
print(f"🎯 目标疾病CUI: {TARGET_CUIS}")

INPUT_FILE = "./mimic_notes_final_clean.csv"
df = pd.read_csv(INPUT_FILE)
df['original_index'] = df.index
text_column = df.columns[0] 
texts = df[text_column].dropna().astype(str).tolist()
print(f"📄 共载入 {len(texts)} 条病历")

START_INDEX = 11999 
if START_INDEX >= len(texts):
    print(f"❌ 错误：起始索引 {START_INDEX} 超出病历总数 {len(texts)}")
    exit()
print(f"⏩ 将从第 {START_INDEX + 1} 条病历开始处理（跳过前 {START_INDEX} 条）")

matched_records = [] 
SAVE_EVERY = 1000
OUTPUT_CSV = "./copd_matched_records1.csv"
OUTPUT_EXCEL = "./copd_matched_records1.xlsx"

print("🔍 开始提取包含'慢性阻塞性气道疾病'的病历...")

for i, text in enumerate(tqdm(texts[START_INDEX:], desc="Processing notes", initial=START_INDEX), start=START_INDEX+1):
    doc = nlp(text[:3000])  
    contains_copd = False
    matched_cuis = set()
    
    for ent in doc.ents:
        if ent.label_ == "DISEASE":
            if ent._.kb_ents:
                cui, score = ent._.kb_ents[0]
                if cui in TARGET_CUIS:
                    contains_copd = True
                    matched_cuis.add(cui)
    if contains_copd:
        original_row = df.iloc[i-1]  
        matched_records.append({
            "row_id": original_row['original_index'],
            "matched_cuis": ", ".join(sorted(matched_cuis)),
            "text_preview": text[:200] + "..." if len(text) > 200 else text,
            "full_text": text,
        })
    if (i - START_INDEX) % SAVE_EVERY == 0 or i == len(texts):
        if matched_records:
            temp_df = pd.DataFrame(matched_records)
            temp_df.to_csv(OUTPUT_CSV, index=False)
            print(f"\n💾 已保存进度：{i}/{len(texts)} 条病历 → 匹配到 {len(matched_records)} 条")

if matched_records:
    result_df = pd.DataFrame(matched_records)
    
    result_df.to_csv(OUTPUT_CSV, index=False)
    result_df.to_excel(OUTPUT_EXCEL, index=False, engine='openpyxl')
    
    print(f"\n✅ 提取完成！")
    print(f"📊 从第 {START_INDEX+1} 条开始，共找到 {len(result_df)} 条包含'慢性阻塞性气道疾病'的病历")
    print(f"📁 结果已保存：")
    print(f"   - {OUTPUT_CSV} (CSV格式)")
    print(f"   - {OUTPUT_EXCEL} (Excel格式，含完整文本)")
    
    print("\n🔬 匹配到的UMLS CUI分布：")
    cui_counts = {}
    for record in matched_records:
        for cui in record['matched_cuis'].split(", "):
            cui_counts[cui] = cui_counts.get(cui, 0) + 1
    
    for cui, count in cui_counts.items():
        try:
            canonical_name = linker.kb.cui_to_entity[cui].canonical_name
        except:
            canonical_name = "未知概念"
        print(f"  {cui}: {canonical_name} - {count} 次")
    
    print(f"\n🔍 前3条匹配结果预览：")
    for idx, record in enumerate(result_df.head(3).itertuples(), 1):
        print(f"\n【结果 {idx}】")
        print(f"CUI: {record.matched_cuis}")
        print(f"文本预览: {record.text_preview}")
        print("-" * 50)
else:
    print(f"\n⚠️ 从第 {START_INDEX+1} 条开始，未找到任何包含目标疾病的病历。")